# 🤖 AI Agent com SmolAgents

##1. Introdução

Este notebook foi desenvolvido durante o **Hugging Face AI Agents Course** (Bootcamp triggo.ai).

**Objetivos**

- Aprender a utilizar o framework **SmolAgents**;
- Criar um agente baseado em LLM;
- Integrar ferramentas (Tools);
- Realizar pesquisas na Web;
- Gerar imagens a partir de prompts;
- Disponibilizar o agente através de uma interface Gradio.

**Tecnologias**

- Python
- SmolAgents
- Hugging Face Hub
- Gradio
- Qwen2.5-Coder-32B-Instruct

##2. Instalação de Bibliotecas

**INSTALANDO AS DEPENDÊNCIAS**

Antes de construir o agente, precisamos instalar as bibliotecas que serão utilizadas durante o desenvolvimento.

Neste projeto utilizaremos:

- **smolagents:** framework para criação de agentes
- **gradio:** interface web para interação
- **huggingface_hub:** acesso aos modelos hospedados no Hub
- **pyyaml:** leitura de arquivos YAML

In [1]:
# Instalação das bibliotecas necessárias.
# O parâmetro -q reduz a quantidade de mensagens exibidas.

!pip install -q smolagents gradio huggingface-hub pyyaml

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

##3. Autenticação no HuggingFace

**AUTENTICAÇÃO**

O agente utilizará modelos hospedados no Hugging Face Hub.

Para isso é necessário autenticar utilizando um Access Token.

A autenticação permite:

- acessar modelos privados (quando necessário);
- utilizar Inference Endpoints;
- carregar Tools publicadas pela comunidade.

In [ ]:
# Autenticação na Hugging Face.
# Necessária para utilizar modelos hospedados no Hub.

from huggingface_hub import notebook_login

notebook_login()

##4. Imports

**IMPORTANDO AS BIBLIOTECAS**

Nesta etapa importamos todas as dependências necessárias.

Cada biblioteca possui uma responsabilidade específica:

- CodeAgent → agente principal
- Tool → definição de ferramentas
- WebSearchTool → pesquisas na internet
- InferenceClientModel → comunicação com o LLM
- GradioUI → interface gráfica

In [ ]:
# Importação das bibliotecas utilizadas pelo agente.

from smolagents import (
    CodeAgent,
    InferenceClientModel,
    tool,
    WebSearchTool
)
from smolagents.gradio_ui import GradioUI

import datetime
import pytz

##5. Tools
*   Consulta de Horário
*   Geração de Imagem



**O QUE SÃO AS TOOLS?**

Um LLM possui conhecimento, mas não consegue executar ações sozinho.

As **Tools** permitem que o agente interaja com o mundo externo.

Exemplos:

- consultar horário
- pesquisar na internet
- acessar banco de dados
- chamar APIs
- gerar imagens
- enviar emails

Durante o raciocínio, o agente decide autonomamente quando utilizar cada Tool.

### Tool: Consulta de Horário

Esta Tool retorna o horário atual de qualquer fuso horário informado.

Embora um LLM saiba o conceito de horário, ele não conhece o horário atual.

Por isso precisamos fornecer essa capacidade através de uma Tool.

In [ ]:
# Ferramenta responsável por retornar o horário atual
# de qualquer fuso informado pelo usuário.
#
# Exemplo:
# "America/Sao_Paulo"
# "Europe/London"
# "Asia/Tokyo"

@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """A tool that fetches the current local time in a specified timezone.
    Args:
        timezone: A string representing a valid timezone (e.g., 'America/New_York').
    """
    try:
        # Create timezone object
        tz = pytz.timezone(timezone)
        # Get current time in that timezone
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"The current local time in {timezone} is: {local_time}"
    except Exception as e:
        return f"Error fetching time for timezone '{timezone}': {str(e)}"


### Tool: Geração de Imagens

Além de responder perguntas, um agente também pode executar tarefas multimodais.

Nesta etapa utilizaremos uma Tool publicada no Hugging Face Hub capaz de gerar imagens a partir de prompts em linguagem natural.

In [ ]:
from huggingface_hub import InferenceClient

# O decorator @tool transforma uma função Python
# em uma ferramenta que poderá ser utilizada pelo agente
# durante seu processo de raciocínio.

@tool
def generate_image(prompt: str) -> object:
    """
    Generates an image based on a provided text prompt.

    Args:
        prompt: The text description of the image you want to generate.
    """
    client = InferenceClient("black-forest-labs/FLUX.1-schnell")
    return client.text_to_image(prompt)

In [ ]:
# Carrega uma ferramenta publicada no Hugging Face Hub.
# Neste exemplo é utilizada uma Tool pronta para geração
# de imagens, disponibilizada pelo curso.

from smolagents import load_tool

image_generation_tool = load_tool("agents-course/text-to-image", trust_remote_code=True)

tool.py:   0%|          | 0.00/635 [00:00<?, ?B/s]

##6. Configuração do Modelo de IA

**CONFIGURANDO O LLM**

O LLM funciona como o "cérebro" do agente.

Ele é responsável por:

- interpretar a solicitação do usuário;
- decidir quais Tools utilizar;
- gerar o raciocínio;
- produzir a resposta final.

Neste projeto utilizaremos o modelo:

**Qwen2.5-Coder-32B-Instruct**

In [ ]:
# Configuração do modelo de linguagem que será utilizado
# pelo agente para raciocinar e executar tarefas.

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",  # Define qual LLM será utilizado
    max_tokens=2048,                             # Define o tamanho máximo da resposta
    temperature=0.5                              # Controla a Criatividade do modelo
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a9e260577c353f1bf4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ gere uma imagem realista do Homem de Ferro                                                                      │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  prompt = "A high-res, photorealistic image of Iron Man, in his iconic red and gold armor, standing in a modern   
  urban environment with a gritty, realistic texture. The lighting is dramatic, casting deep shadows and           
  highlighting the metallic surfaces of his suit. High detail on the armor plating and a sense of movement or      
  action to make it more dynamic."                                                                                 
  image = image_generator(prompt=prompt)                                                                           
  final_answer(image)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1024x1024 at 0x7C49863A8530>

[Step 1: Duration 7.95 seconds| Input tokens: 2,211 | Output tokens: 129]

##7. Construção do Agente

**CRIANDO O AGENTE**

Agora reunimos todos os componentes.

O agente será composto por:

- um LLM
- um conjunto de Tools
- um mecanismo de raciocínio

Sempre que receber uma pergunta, o agente seguirá um ciclo semelhante ao abaixo:

Usuário

↓

LLM analisa a solicitação

↓

Precisa utilizar alguma Tool?

↓

Sim → Executa a Tool

↓

Recebe o resultado

↓

Continua o raciocínio

↓

Resposta final

In [ ]:
# O CodeAgent é responsável por decidir,
# durante a execução, quando utilizar cada Tool.

agent=CodeAgent(
    tools=[
        get_current_time_in_timezone,   # Consulta horário
        WebSearchTool(),                # Pesquisa na internet
        image_generation_tool           # Geração de imagens
    ]
    model=model,
    max_steps=6)

##8. Inicialização da Interface Gradio

**INTERFACE DO USUÁRIO**

Após construir o agente, disponibilizamos uma interface web utilizando Gradio.

Isso permite interagir com o agente diretamente pelo navegador sem necessidade de executar chamadas Python manualmente.

In [ ]:
# Inicializa uma interface web simples para interação
# com o agente diretamente pelo navegador.

GradioUI(agent).launch()

## Conclusão

Neste notebook aprendemos como construir um agente utilizando o framework SmolAgents.

Conceitos abordados:

✅ LLMs

✅ AI Agents

✅ Tools

✅ Function Calling

✅ Web Search

✅ Geração de Imagens

✅ Gradio

Próximos passos:

- Memória de longo prazo
- Múltiplos agentes
- LangGraph
- MCP
- Agentic RAG